# X3D-M M3A component evaluation

This Colab-only notebook evaluates the pinned pretrained X3D-M violence adapter against the Drive binary manifests. It reports held-out results per dataset; crowd features, fusion, incidents, and training remain out of scope.

In [ ]:
from collections import defaultdict
from pathlib import Path, PurePosixPath
from dataclasses import replace
import hashlib, json, os, re, subprocess, sys, tempfile

DRIVE_ROOT = Path('/content/drive/MyDrive/crowd_safety')
MANIFEST_ROOT = DRIVE_ROOT / 'manifests'
EVALUATION_ROOT = DRIVE_ROOT / 'evaluation/runs/x3d_m3a_component_v2'
PROJECT_ROOT = Path('/content/realtime-crowd-safety-monitoring')
RUN_EVALUATION = False
REUSE_EXISTING_RUN = True
EVAL_LIMIT = 300

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass
if (Path.cwd() / 'src').is_dir():
    PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    if PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f'Project path exists but has no src directory: {PROJECT_ROOT}')
    clone = subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/govardhan-06/realtime-crowd-safety-monitoring.git', str(PROJECT_ROOT)], capture_output=True, text=True)
    if clone.returncode != 0:
        raise RuntimeError(f'Could not clone project: {clone.stderr.strip()}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{PROJECT_ROOT}[violence]'], check=True)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

X3D_CONTRACT = {
    'repository': 'visionlab-ai/school-violence-detection-models',
    'checkpoint': 'final/final_x3d_realtime.pt',
    'revision': 'a744b6af7496f0cbfa4f0ba32acd46b65e52d4e1',
    'checkpoint_sha256': 'e833f69d110f167cad4a6c38d385564bdb2f6de63d246e45cb03ff9aa17f0349',
    'architecture': 'x3d_m', 'sample_count': 16, 'frame_size': [224, 224],
    'mean': [0.45, 0.45, 0.45], 'std': [0.225, 0.225, 0.225],
    'labels': ['non-violent', 'violent'],
}

def validate_x3d_contract(contract):
    required = {'repository', 'checkpoint', 'revision', 'checkpoint_sha256', 'architecture', 'sample_count', 'frame_size', 'mean', 'std', 'labels'}
    if not required <= contract.keys() or not re.fullmatch(r'[0-9a-f]{40}', contract['revision']) or not re.fullmatch(r'[0-9a-f]{64}', contract['checkpoint_sha256']):
        raise ValueError('X3D contract is incomplete or unpinned')
    if contract['architecture'] != 'x3d_m' or contract['sample_count'] != 16 or contract['frame_size'] != [224, 224] or contract['labels'] != ['non-violent', 'violent']:
        raise ValueError('X3D architecture, input, or labels changed')
    return True

validate_x3d_contract(X3D_CONTRACT)
print({'manifest_root': str(MANIFEST_ROOT), 'evaluation_root': str(EVALUATION_ROOT), 'threshold': 0.4, 'eval_limit': EVAL_LIMIT})

## Load the Drive binary manifests

Only `test.json` and `external_test.json` are scored. The fixed threshold comes from the development config; no external-test calibration is performed.

In [ ]:
MANIFEST_FILES = {'train': 'train.json', 'validation': 'val.json', 'test': 'test.json', 'external_test': 'external_test.json'}
DATASETS = {'violent_flows', 'ubi_fights', 'surveillance_fight', 'scvd'}

def resolve_binary_media(record, require_exists=False):
    relative = record['relative_path']; path = PurePosixPath(relative)
    if path.is_absolute() or '..' in path.parts or not relative.startswith('datasets/'):
        raise ValueError('unsafe Drive-relative media path: ' + relative)
    root = DRIVE_ROOT.resolve(); candidate = (root / relative).resolve()
    try:
        candidate.relative_to(root)
    except ValueError as exc:
        raise ValueError('media path escapes Drive root: ' + relative) from exc
    if require_exists and not candidate.is_file():
        raise FileNotFoundError('media missing: ' + str(candidate))
    return candidate

def load_binary_manifests(require_files=False):
    records, paths, hashes = [], set(), set()
    for split, filename in MANIFEST_FILES.items():
        path = MANIFEST_ROOT / filename
        if not path.is_file():
            raise FileNotFoundError('missing binary manifest: ' + str(path))
        payload = json.loads(path.read_text())
        if payload.get('manifest_type') != 'binary_video' or payload.get('split') != split:
            raise ValueError('invalid binary manifest: ' + str(path))
        for record in payload.get('records', []):
            if not isinstance(record, dict) or record.get('dataset') not in DATASETS or record.get('label') not in {'normal', 'violent'} or record.get('split') != split:
                raise ValueError('unsupported binary record: ' + repr(record))
            resolve_binary_media(record, require_files)
            digest = record.get('sha256')
            if record['relative_path'] in paths or (digest and digest in hashes):
                raise ValueError('duplicate path/hash across binary manifests')
            paths.add(record['relative_path']); hashes.add(digest) if digest else None
            if split == 'external_test' and record['dataset'] != 'violent_flows':
                raise ValueError('external_test must contain only violent_flows')
            if split != 'external_test' and record['dataset'] == 'violent_flows':
                raise ValueError('violent_flows cannot be train/validation/test')
            records.append(record)
    return records

def binary_self_check():
    fixture = [{'dataset': 'scvd', 'relative_path': 'datasets/scvd/a.mp4', 'label': 'violent', 'split': 'test'}]
    assert fixture[0]['dataset'] in DATASETS and resolve_binary_media(fixture[0]) == DRIVE_ROOT / 'datasets/scvd/a.mp4'
    try:
        resolve_binary_media({**fixture[0], 'relative_path': '../secret.mp4'})
    except ValueError: pass
    else: raise AssertionError('unsafe path accepted')
    print('media-independent X3D manifest self-check: ok')

binary_self_check()

## Score saved or fresh X3D windows

In [ ]:
def jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.is_file() else []

def validate_reusable_run(run_directory, record=None):
    run_directory = Path(run_directory)
    metadata = json.loads((run_directory / 'metadata.json').read_text())
    actual = metadata.get('provenance', {}).get('violence_provenance')
    expected = {key: X3D_CONTRACT[key] for key in ('repository', 'checkpoint', 'revision', 'architecture', 'sample_count', 'checkpoint_sha256')}
    expected.update({'backend': 'x3d', 'labels': X3D_CONTRACT['labels']})
    if actual:
        if any(actual.get(key) != value for key, value in expected.items()):
            raise ValueError(f'saved run provenance does not match pinned X3D contract: actual={actual!r}')
    else:
        metrics = json.loads((run_directory / 'metrics.json').read_text())
        violence_health = metrics.get('stage_health', {}).get('violence', {})
        if violence_health.get('status') not in {'available', 'degraded', 'unavailable'}:
            raise ValueError('saved run has no X3D provenance and no valid violence stage status')
        if jsonl(run_directory / 'violence.jsonl'):
            raise ValueError('saved run has violence evidence but no provenance')
        print(f'reusing completed run with no violence evidence ({violence_health.get("status")}) : {run_directory}', flush=True)
    if record is not None:
        config = json.loads((run_directory / 'config.json').read_text()).get('config', {})
        stored_input = config.get('input_path')
        if not isinstance(stored_input, str) or Path(stored_input).expanduser().resolve() != resolve_binary_media(record):
            raise ValueError('saved run input does not match the manifest media')

def video_result_slug(record):
    stem = re.sub(r'[^A-Za-z0-9._-]+', '_', Path(record['relative_path']).stem).strip('._') or 'video'
    digest = hashlib.sha1(record['relative_path'].encode('utf-8')).hexdigest()[:8]
    return f'{stem}-{digest}'

def run_has_reusable_violence_result(run_directory):
    run_directory = Path(run_directory)
    evidence_rows = jsonl(run_directory / 'violence.jsonl')
    if evidence_rows:
        return any((row.get('evidence', row).get('status') == 'available' and row.get('evidence', row).get('score') is not None) for row in evidence_rows)
    metrics = json.loads((run_directory / 'metrics.json').read_text())
    return metrics.get('stage_health', {}).get('violence', {}).get('status') == 'available'

def validate_runtime_violence_config(violence_config):
    expected = {key: X3D_CONTRACT[key] for key in ('repository', 'checkpoint', 'revision', 'architecture', 'sample_count', 'checkpoint_sha256', 'labels')}
    actual = {key: getattr(violence_config, key, None) for key in expected}
    actual['labels'] = list(actual['labels']) if actual['labels'] is not None else None
    mismatches = {key: (actual[key], value) for key, value in expected.items() if actual[key] != value}
    if getattr(violence_config, 'backend', None) != 'x3d':
        mismatches['backend'] = (getattr(violence_config, 'backend', None), 'x3d')
    if mismatches:
        raise RuntimeError(f'Colab project checkout is not the pinned X3D M3A implementation: {mismatches}')
    return True

def discover_existing_run_map(records):
    candidates = {}
    run_directories = {path.parent for path in EVALUATION_ROOT.rglob('metadata.json')} if EVALUATION_ROOT.is_dir() else set()
    print(f'reuse scan: found {len(run_directories)} run directories', flush=True)
    required = ('metadata.json', 'config.json', 'violence.jsonl', 'metrics.json', 'annotated.mp4')
    incomplete = 0
    for run_directory in sorted(run_directories):
        if not all((run_directory / name).is_file() for name in required):
            incomplete += 1
            continue
        try:
            validate_reusable_run(run_directory)
            if not run_has_reusable_violence_result(run_directory):
                continue
            config = json.loads((run_directory / 'config.json').read_text()).get('config', {})
            stored_input = Path(config['input_path']).expanduser().resolve()
        except (KeyError, OSError, TypeError, ValueError, json.JSONDecodeError):
            continue
        candidates[stored_input] = run_directory
    matched = {record['relative_path']: candidates[resolve_binary_media(record)] for record in records if resolve_binary_media(record) in candidates}
    print(f'reuse scan: complete={len(candidates)} matched={len(matched)} incomplete={incomplete}', flush=True)
    return matched
def score_rows(rows, threshold):
    available = [row for row in rows if row.get('status') == 'available' and row.get('score') is not None]
    tp = sum(row['label'] == 'violent' and float(row['score']) >= threshold for row in available)
    tn = sum(row['label'] == 'normal' and float(row['score']) < threshold for row in available)
    fp = sum(row['label'] == 'normal' and float(row['score']) >= threshold for row in available)
    fn = sum(row['label'] == 'violent' and float(row['score']) < threshold for row in available)
    total = len(available)
    return {'status': 'available' if available else 'pending-compatible-scores', 'samples': total, 'unavailable_windows': len(rows) - total, 'threshold': threshold, 'threshold_source': 'development_config_only', 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0, 'accuracy': (tp + tn) / total if total else None, 'confusion_matrix': [[tn, fp], [fn, tp]] if available else None, 'score_distribution': {'min': min((float(row['score']) for row in available), default=None), 'max': max((float(row['score']) for row in available), default=None), 'mean': sum(float(row['score']) for row in available) / total if total else None}, 'false_positives': [row['video_id'] for row in available if row['label'] == 'normal' and float(row['score']) >= threshold], 'false_negatives': [row['video_id'] for row in available if row['label'] == 'violent' and float(row['score']) < threshold], 'latency_ms_mean': sum(float(row['latency_ms']) for row in available if row.get('latency_ms') is not None) / sum(row.get('latency_ms') is not None for row in available) if any(row.get('latency_ms') is not None for row in available) else None}

def score_by_dataset(rows, threshold):
    grouped = defaultdict(list)
    for row in rows:
        grouped[(row['dataset'], row['split'])].append(row)
    return {f'{dataset}:{split}': {'dataset': dataset, 'split': split, **score_rows(items, threshold)} for (dataset, split), items in sorted(grouped.items())}

def select_diverse_records(records):
    grouped = {dataset: sorted((record for record in records if record['dataset'] == dataset), key=lambda row: row['relative_path']) for dataset in sorted(DATASETS)}
    selected, indexes = [], {dataset: 0 for dataset in grouped}
    while any(indexes[dataset] < len(grouped[dataset]) for dataset in grouped):
        for dataset in sorted(grouped):
            index = indexes[dataset]
            if index < len(grouped[dataset]):
                selected.append(grouped[dataset][index]); indexes[dataset] += 1
    return selected

def rows_from_run(record, run_directory, threshold):
    run_directory = Path(run_directory)
    metadata = json.loads((run_directory / 'metadata.json').read_text())
    provenance = metadata.get('provenance') or {}
    rows = [{
        'video_id': record['relative_path'], 'dataset': record['dataset'], 'label': record['label'],
        'split': record['split'], 'clip_start_s': (evidence := raw.get('evidence', raw)).get('clip_start_s'),
        'clip_end_s': evidence.get('clip_end_s'), 'score': evidence.get('score'),
        'status': evidence.get('status', 'unavailable'), 'latency_ms': evidence.get('latency_ms'),
        'threshold': threshold, 'checkpoint': provenance.get('violence_model', evidence.get('model')),
        'revision': provenance.get('violence_revision', evidence.get('revision')),
    } for raw in jsonl(run_directory / 'violence.jsonl')]
    if rows:
        return rows
    health = json.loads((run_directory / 'metrics.json').read_text()).get('stage_health', {}).get('violence', {})
    health_status = health.get('status', 'unavailable')
    return [{
        'video_id': record['relative_path'], 'dataset': record['dataset'], 'label': record['label'],
        'split': record['split'], 'clip_start_s': None, 'clip_end_s': None, 'score': None,
        'status': 'insufficient' if health_status == 'available' else health_status,
        'latency_ms': health.get('latency_ms'), 'threshold': threshold,
        'checkpoint': provenance.get('violence_model', health.get('model')),
        'revision': provenance.get('violence_revision'), 'detail': health.get('detail'),
    }]

def write_partial_checkpoint(rows, threshold):
    EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
    (EVALUATION_ROOT / 'predictions.partial.jsonl').write_text(''.join(json.dumps(row, sort_keys=True) + '\n' for row in rows))
    partial = {'model': 'M3A', 'contract': X3D_CONTRACT, 'threshold': threshold, 'status': 'partial', 'by_dataset': score_by_dataset(rows, threshold)}
    (EVALUATION_ROOT / 'metrics.partial.json').write_text(json.dumps(partial, indent=2, sort_keys=True) + '\n')

def collect_predictions(records, run_map=None):
    from crowd_safety.config import load_config
    from crowd_safety.runner import process_video
    base = load_config(PROJECT_ROOT / 'configs/pipeline/dev.toml'); validate_runtime_violence_config(base.violence); threshold = base.violence.threshold
    print(f'evaluation backend: {base.violence.backend} | evaluation root: {EVALUATION_ROOT}', flush=True)
    run_map = run_map if run_map is not None else (discover_existing_run_map(records) if REUSE_EXISTING_RUN else {}); rows = []
    selected_all = select_diverse_records(record for record in records if record['split'] in {'test', 'external_test'})
    pending = [record for record in selected_all if record['relative_path'] not in run_map]
    end_index = None if EVAL_LIMIT is None else EVAL_LIMIT
    selected = pending[:end_index]
    print(f'reuse scan: completed={len(selected_all) - len(pending)} pending={len(pending)} processing={len(selected)}', flush=True)
    partial_rows = jsonl(EVALUATION_ROOT / 'predictions.partial.jsonl')
    known_ids = {record['relative_path'] for record in selected_all if record['relative_path'] in run_map}
    rows.extend(row for row in partial_rows if row.get('video_id') in known_ids)
    for record in selected_all:
        if record['relative_path'] in known_ids and not any(row.get('video_id') == record['relative_path'] for row in rows):
            validate_reusable_run(run_map[record['relative_path']], record)
            rows.extend(rows_from_run(record, run_map[record['relative_path']], threshold))
    print(f'reusing {len(known_ids)} completed runs', flush=True)
    for index, record in enumerate(selected, start=1):
        run_directory = None
        if run_directory is None:
            media = resolve_binary_media(record, require_exists=True)
            config = replace(base, input_path=media, output_directory=EVALUATION_ROOT / 'source_runs' / video_result_slug(record))
            print(f'[{index}/{len(selected)}] processing {record["relative_path"]} -> {video_result_slug(record)}', flush=True)
            run_directory = process_video(config, input_override=media).run_directory
        validate_reusable_run(run_directory, record)
        rows.extend(rows_from_run(record, run_directory, threshold))
        write_partial_checkpoint(rows, threshold)
        print(f'checkpointed {index}/{len(selected)} videos', flush=True)
    return rows, threshold

fixture_rows = [{'video_id': 'normal', 'dataset': 'scvd', 'label': 'normal', 'split': 'test', 'score': 0.1, 'status': 'available'}, {'video_id': 'violent', 'dataset': 'ubi_fights', 'label': 'violent', 'split': 'test', 'score': 0.9, 'status': 'available'}, {'video_id': 'missing', 'dataset': 'violent_flows', 'label': 'violent', 'split': 'external_test', 'score': None, 'status': 'unavailable'}]
fixture_metrics = score_by_dataset(fixture_rows, 0.4)
assert fixture_metrics['scvd:test']['accuracy'] == 1.0 and fixture_metrics['ubi_fights:test']['recall'] == 1.0 and fixture_metrics['violent_flows:external_test']['status'] == 'pending-compatible-scores'
assert video_result_slug({'relative_path': 'datasets/scvd/fight clip.mp4'}).startswith('fight_clip-')
assert video_result_slug({'relative_path': 'datasets/ubi_fights/fight clip.mp4'}) != video_result_slug({'relative_path': 'datasets/scvd/fight clip.mp4'})
try:
    validate_runtime_violence_config(type('Config', (), {'backend': 'huggingface'})())
except RuntimeError: pass
else: raise AssertionError('non-X3D runtime config accepted')
with tempfile.TemporaryDirectory() as directory:
    short_run = Path(directory)
    (short_run / 'metadata.json').write_text(json.dumps({'provenance': {'violence_provenance': {}}}))
    (short_run / 'metrics.json').write_text(json.dumps({'stage_health': {'violence': {'status': 'available'}}}))
    (short_run / 'violence.jsonl').write_text('')
    validate_reusable_run(short_run)
    (short_run / 'metrics.json').write_text(json.dumps({'stage_health': {'violence': {'status': 'unavailable'}}}))
    validate_reusable_run(short_run)
    assert not run_has_reusable_violence_result(short_run)
    summary = rows_from_run({'relative_path': 'datasets/scvd/short.mp4', 'dataset': 'scvd', 'label': 'normal', 'split': 'test'}, short_run, 0.4)
    assert summary[0]['score'] is None and summary[0]['status'] == 'unavailable'
    assert score_rows(summary, 0.4)['samples'] == 0 and score_rows(summary, 0.4)['unavailable_windows'] == 1
    (short_run / 'metrics.json').write_text(json.dumps({'stage_health': {'violence': {'status': 'available'}}}))
    assert run_has_reusable_violence_result(short_run)
    (short_run / 'violence.jsonl').write_text('{\"evidence\": {}}\n')
    try:
        validate_reusable_run(short_run)
    except ValueError: pass
    else: raise AssertionError('evidence without provenance accepted')
diversity_fixture = [{'dataset': dataset, 'relative_path': f'datasets/{dataset}/{index}.mp4'} for dataset in DATASETS for index in range(2)]
assert [row['dataset'] for row in select_diverse_records(diversity_fixture)[:4]] == sorted(DATASETS)
print('per-dataset scoring self-check: ok')

## Execute and save the component report

Set `RUN_EVALUATION = True` for an authorised Drive run. `EVAL_LIMIT = 300` processes up to 300 pending videos in deterministic round-robin order; rerun the cell to continue with the remaining pending videos. Each result is grouped under a readable `<original-stem>-<stable-path-hash>` directory. Completed matching runs on Drive are reused automatically; partial predictions and metrics are checkpointed after every video in `predictions.partial.jsonl` and `metrics.partial.json`. Optionally provide `CROWD_SAFETY_M3A_RUN_MAP` as JSON to override the discovered map.

In [ ]:
if RUN_EVALUATION:
    records = load_binary_manifests(require_files=True)
    run_map = json.loads(os.environ['CROWD_SAFETY_M3A_RUN_MAP']) if os.environ.get('CROWD_SAFETY_M3A_RUN_MAP') else None
    predictions, threshold = collect_predictions(records, run_map)
    report = {'model': 'M3A', 'contract': X3D_CONTRACT, 'threshold': threshold, 'threshold_source': 'development_config_only; external_test was not calibrated', 'by_dataset': score_by_dataset(predictions, threshold)}
    EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
    (EVALUATION_ROOT / 'predictions.jsonl').write_text(''.join(json.dumps(row, sort_keys=True) + '\n' for row in predictions))
    (EVALUATION_ROOT / 'metrics.json').write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
    print(json.dumps(report, indent=2, sort_keys=True))
else:
    print('self-check mode; set RUN_EVALUATION = True for authorised Drive execution')